# GNN-Based BERT for Understanding Context from Music
### CSE425 Supervised Neural Network Project — End-to-End Demo Notebook

This notebook demonstrates an end-to-end inference pipeline combining **Graph Neural Networks (GNN)** on music structure graphs with **BERT** on natural-language contextual descriptions to predict multi-label musical genres.

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('src'))

import features_store
import graph_builder as gb
from labels import build_vocab
from fusion_model import FusionModel
from transformers import BertTokenizerFast

print('Environment initialized successfully!')

## 1. Load Audio Track Features (Person 1 & 3)
We load the 128-bin log-mel spectrogram and 12-bin chroma features for track `000181` from our preprocessed feature store.

In [ ]:
track_id = '000181'
mel, chroma = features_store.get(track_id)
print('Track ID:', track_id)
print('Segmented Mel Spectrogram shape:', mel.shape)
print('Segmented Chroma STFT shape:', chroma.shape)

# Plot spectrogram and chroma
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
mel_stitch, chroma_stitch = features_store.stitch(track_id)
axes[0].imshow(mel_stitch, aspect='auto', origin='lower', cmap='magma')
axes[0].set_title(f'Track {track_id} - Log-Mel Spectrogram (128 Bins)')
axes[0].set_ylabel('Mel Frequency')

axes[1].imshow(chroma_stitch, aspect='auto', origin='lower', cmap='coolwarm')
axes[1].set_title(f'Track {track_id} - Chroma Features (12 Pitch Classes)')
axes[1].set_ylabel('Pitch Class')
axes[1].set_xlabel('Time Frame')
plt.tight_layout()
plt.show()

## 2. Construct Music Structure Graph (Person 3)
We segment the track into 60 temporal windows and establish edges based on temporal adjacency and k-nearest-neighbor (k=3) acoustic similarity in the mel+chroma feature space.

In [ ]:
n_nodes = 60
raw_nodes = gb.node_features_batch(mel_stitch[None], chroma_stitch[None], n_nodes=n_nodes, feat='mel+chroma', with_std=True)[0]
norm_nodes = ((raw_nodes - raw_nodes.mean(0)) / (raw_nodes.std(0) + 1e-6)).astype(np.float32)
edge_index, edge_attr = gb.build_edges(norm_nodes, mode='knn', k=3, tau=0.7)

print('Music Graph constructed:')
print('  Number of Segment Nodes:', norm_nodes.shape[0])
print('  Node Feature Dimension:', norm_nodes.shape[1])
print('  Number of Structural Edges:', edge_index.shape[1])

## 3. Natural Language Contextual Description & BERT Tokenization (Person 2)
We prepare the semantic text description incorporating artist, album, and stylistic descriptors, then tokenize it for BERT.

In [ ]:
text_desc = "A Rock music piece titled 'Gopacapulco' by Ariel Pink's Haunted Graffiti from the album 'Scared Famous'."
print('Text Description:', text_desc)

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
tokens = tokenizer(text_desc, padding='max_length', truncation=True, max_length=64, return_tensors='pt')
print('Tokenized Input IDs shape:', tokens['input_ids'].shape)

## 4. End-to-End Inference with GNN-BERT Cross-Attention Fusion (Person 4)
We load the trained multi-modal fusion model checkpoint (`results/fusion/fusion_cross_attention.pt`) and perform end-to-end inference.

In [ ]:
vocab = build_vocab('multi', top_k=20)
ckpt_path = '../results/fusion/fusion_cross_attention.pt' if os.path.exists('../results/fusion/fusion_cross_attention.pt') else 'results/fusion/fusion_cross_attention.pt'

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = FusionModel(num_labels=len(vocab), gnn_in_dim=280, mode='cross_attention')
model.load_state_dict(ckpt['model_state'])
model.eval()

x_tensor = torch.from_numpy(norm_nodes)
batch_idx = torch.zeros(n_nodes, dtype=torch.long)

with torch.no_grad():
    logits = model(input_ids=tokens['input_ids'], attention_mask=tokens['attention_mask'], x=x_tensor, edge_index=edge_index, batch=batch_idx)
    probabilities = torch.sigmoid(logits)[0].numpy()

print('Inference complete! Multi-label probabilities computed.')

## 5. Predicted Genres vs. Ground Truth
We rank and display the model's top predicted genre affinities.

In [ ]:
top_k = 5
ranked_indices = np.argsort(probabilities)[::-1][:top_k]

print('=== Top Predicted Genres ===')
for rank, idx in enumerate(ranked_indices, 1):
    print(f'{rank}. {vocab[idx]:<20} Confidence: {probabilities[idx]*100:.2f}%')

# Visual bar chart
plt.figure(figsize=(8, 4))
plt.barh([vocab[i] for i in ranked_indices[::-1]], [probabilities[i] for i in ranked_indices[::-1]], color='teal')
plt.xlabel('Prediction Probability')
plt.title(f'Model Confidence for Track {track_id}')
plt.xlim(0, 1.0)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()